# 1. Load TMDB dataset using MyPyTable

In [7]:
from mysklearn.mypytable import MyPyTable
from mysklearn.utils import get_main_genre
import os

os.makedirs("output_data", exist_ok=True)

# load raw TMDB dataset
tmdb_raw = MyPyTable().load_from_file("input_data/tmdb_5000_movies.csv")

print("Original shape:", tmdb_raw.get_shape())
print("Columns:", tmdb_raw.column_names)


Original shape: (4803, 20)
Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


# 2. Select needed columns


In [8]:
needed_columns = [
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "vote_count",
    "release_date",
    "genres"
]

col_indexes = {}
for col in needed_columns:
    col_indexes[col] = tmdb_raw.column_names.index(col)

print("Selected columns:", needed_columns)


Selected columns: ['budget', 'popularity', 'runtime', 'vote_average', 'vote_count', 'release_date', 'genres']


# 3. Build a clean table in MyPyTable style

In [9]:
def clean_num(val):
    """Turn empty or weird numeric values into 'NA'."""
    if val == "" or val is None:
        return "NA"
    return val

clean_header = [
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "vote_count",
    "release_year",
    "main_genre"
]

clean_rows = []

for row in tmdb_raw.data:
    budget = clean_num(row[col_indexes["budget"]])
    pop = clean_num(row[col_indexes["popularity"]])
    runtime = clean_num(row[col_indexes["runtime"]])
    vote_avg = clean_num(row[col_indexes["vote_average"]])
    vote_cnt = clean_num(row[col_indexes["vote_count"]])

    # release_date -> release_year
    date_value = row[col_indexes["release_date"]]
    if isinstance(date_value, str) and len(date_value) >= 4:
        release_year = int(date_value[:4])
    else:
        release_year = "NA"

    # genres -> main_genre
    genres_value = row[col_indexes["genres"]]
    main_genre = get_main_genre(genres_value)

    # Action / Comedy / Drama
    if main_genre not in ("Action", "Comedy", "Drama"):
        continue   

    clean_rows.append([
        budget, pop, runtime,
        vote_avg, vote_cnt,
        release_year, main_genre
])


# 4. clean missing values

In [10]:
tmdb_step1 = MyPyTable(clean_header, clean_rows)

numeric_columns = [
    "budget", "popularity", "runtime",
    "vote_average", "vote_count",
    "release_year"
]

for col in numeric_columns:
    tmdb_step1.replace_missing_values_with_column_average(col)

print("After cleaning shape:", tmdb_step1.get_shape())

After cleaning shape: (4029, 7)


# 5. Save data 

In [11]:
output_path = "output_data/tmdb_clean_step1.csv"
tmdb_step1.save_to_file(output_path)

print("File saved at:", output_path)


File saved at: output_data/tmdb_clean_step1.csv
